## PX1 / Mini-Cam — полный просмотр состава заводского обновления (без подключения к серверам)

В предыдущем отчёте подтвердилось, что `factoryupdate.zip` содержит 1029 элементов, но были перечислены только первые 80. Этот блокнот проверяет **все 1029 названий** и ищет среди них схемы, CAD, чертежи, инструкции и прошивки. Также проверяет номера версий в трёх локальных `readme.txt`. Полное содержимое `readme.txt`, серверные адреса, пароли и код прошивок **не выводятся и никуда не отправляются**. Выполняется только чтение твоего образа в Google Colab.

На Android нажми **Среда выполнения → Выполнить все**. В Google Диске появится `PX1_SD_ZIP_MANIFEST_REPORT.txt`.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, struct, shutil
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive')
src = root / 'Untitled folder' / 'ЛП.imgc'
if not src.is_file():
    src = next((Path(d)/'ЛП.imgc' for d,_,names in os.walk(root) if 'ЛП.imgc' in names), None)
if not src or not src.is_file(): raise FileNotFoundError('Не найден ЛП.imgc')
project=Path('/content/unimgc')
if not project.is_dir():
    subprocess.run(['git','clone','--depth','1','https://github.com/Shizmob/unimgc.git',str(project)],check=True)
subprocess.run(['make','-C',str(project)],check=True)
part=Path('/content/px1_ro_partition.img')
offset=32901120
size=7912719360
if shutil.disk_usage('/content').free < size + 1024**3:
    raise RuntimeError('Мало свободного места в Colab для копии раздела')
print('Читаю раздел в временный файл Colab (примерно 7.9 ГБ)...')
p=subprocess.Popen([str(project/'unimgc'),str(src)],stdout=subprocess.PIPE,stderr=subprocess.DEVNULL)
try:
    toskip=offset
    while toskip:
        data=p.stdout.read(min(4*1024*1024,toskip))
        if not data: raise EOFError('Образ закончился до начала раздела')
        toskip-=len(data)
    left=size
    with part.open('wb') as out:
        while left:
            data=p.stdout.read(min(4*1024*1024,left))
            if not data: raise EOFError('Образ закончился внутри раздела')
            out.write(data)
            left-=len(data)
finally:
    p.stdout.close()
    p.terminate()
    try: p.wait(timeout=10)
    except subprocess.TimeoutExpired: p.kill(); p.wait()
def dbg(command):
    v=subprocess.run(['debugfs','-R',command,str(part)],capture_output=True,timeout=90)
    return v.stdout.decode('utf-8','replace')

# Read-only inspection of locally owned offline ZIP. No proprietary files are exported.
import zipfile, collections, re
report=[
'PX1 Mini-Cam — complete local factoryupdate.zip filename inventory',
'Only ZIP entry names/counts and whitelisted version-number strings are reported.'
]
def add(k,v): report.append(k+': '+str(v))
archive='/root/factoryupdate.zip'
outlocal=Path('/content/px1_factory_zip_index.zip')
outlocal.unlink(missing_ok=True)
stat=dbg('stat '+archive)
if 'Type: regular' not in stat:
    raise RuntimeError('Не обнаружен обычный архив '+archive)
dbg('dump '+archive+' '+str(outlocal))
if not outlocal.is_file():
    raise RuntimeError('Не получилось извлечь ZIP в хранилище Colab')
interesting_ext={'.pdf','.txt','.doc','.docx','.dwg','.dxf','.idw','.iam','.ipt','.step','.stp','.stl','.iges','.igs','.sch','.brd','.kicad_sch','.kicad_pcb','.pcb','.csv','.xml','.json','.ini'}
cad_ext={'.pdf','.doc','.docx','.dwg','.dxf','.idw','.iam','.ipt','.step','.stp','.stl','.iges','.igs','.sch','.brd','.kicad_sch','.kicad_pcb','.pcb'}
with zipfile.ZipFile(outlocal,'r') as z:
    inf=z.infolist()
    add('Всего элементов',len(inf))
    add('Всего файлов',sum(not i.is_dir() for i in inf))
    add('Размер несжатых файлов',sum(i.file_size for i in inf))
    types=collections.Counter()
    roots=collections.Counter()
    candidates=[]
    update_files=[]
    for i in inf:
        if i.is_dir():continue
        name=i.filename
        roots[name.split('/')[0]]+=1
        ext=Path(name).suffix.lower()
        types[ext or '(нет расширения)']+=1
        if ext in cad_ext or any(t in name.lower() for t in ('schematic','schem','drawing','cad','wiring','circuit','service-manual')):
            # Avoid printing unusual characters or path injection in report.
            if re.fullmatch(r'[A-Za-z0-9_/+ .@()-]{1,180}',name):
                candidates.append(name+' ('+str(i.file_size)+' B)')
        if ext in {'.s19','.srec','.hex','.bin','.elf','.upd','.fw'}:
            if re.fullmatch(r'[A-Za-z0-9_/+ .@()-]{1,180}',name):
                update_files.append(name+' ('+str(i.file_size)+' B)')
    add('Файлы по верхним каталогам',dict(sorted(roots.items())))
    add('Файлы по расширениям (топ 30)',str(types.most_common(30)))
    add('Кандидаты на схемы/чертежи',len(candidates))
    report.extend(['  DRAWING '+x for x in candidates[:100]])
    add('Бинарники или файлы прошивки (по расширению)',len(update_files))
    report.extend(['  FW '+x for x in update_files[:80]])
    if len(candidates)>100:add('Не показано других кандидатов',len(candidates)-100)
    if len(update_files)>80:add('Не показано остальных прошивок',len(update_files)-80)
# Probe only whitelisted version-like strings from local firmware release notes.
for directory in ['~~factory','~~V6-4-6-G13','~~PROTEUS_V4.70']:
    path='/root/Firmware/'+directory+'/readme.txt'
    raw=dbg('cat '+path)
    if not raw or len(raw)>15000:
        add('Readme '+directory,'empty or unavailable');continue
    # Never transmit raw readme lines: text may contain server paths, URLs, credentials.
    vals=set(re.findall(r'\b(?:v|V)?\d{1,3}\.\d{1,3}(?:\.\d{1,5}){0,3}\b',raw))
    add('Readme '+directory+' version-like numbers',', '.join(sorted(vals)[:30]) if vals else 'not found')
out=root/'PX1_SD_ZIP_MANIFEST_REPORT.txt'
out.write_text('\n'.join(report)+'\n',encoding='utf-8')
outlocal.unlink(missing_ok=True)
part.unlink(missing_ok=True)
print('\n'.join(report))
print('ГОТОВО:',out.name)
